# Import Salmon counts

In [ ]:
# Load Packages 
library(tximport)
library(readr)
library(fs)
library(GenomicFeatures)

In [ ]:
#create vector of the file paths where Salmon files are 
quant_files <- dir_ls("./discovery_validation//quant_files/", glob ="*.sf")
length(quant_files)
class(quant_files)

In [ ]:
#check salmon files
quants <- read_tsv(quant_files[1])
head(quants)

In [ ]:
#create mapping file for gene-level counts
#gtf_file <- "Homo_sapiens.GRCh38.109.chr.gtf.gz" 
#download.file("https://ftp.ensembl.org/pub/release-109/gtf/homo_sapiens/Homo_sapiens.GRCh38.109.chr.gtf.gz", 
#              destfile = gtf_file)
#download.file("https://ftp.ensembl.org/pub/release-109/gtf/homo_sapiens/Homo_sapiens.GRCh38.109.abinitio.gtf.gz", 
#              destfile = gtf_file)
#txdb <- makeTxDbFromGFF(gtf_file)
#keytypes(txdb)
#columns(txdb)

#k <- keys(txdb, keytype="TXNAME")
#tx_map <- select(txdb, keys = k, columns="GENEID", keytype = "TXNAME")
#head(tx_map)
#dim(tx_map)

In [ ]:
#read mapping file
tx_map <- read.csv("Fisch_tx2gene.csv", header = FALSE)
head(tx_map)

In [ ]:
#import

txi.tx <- tximport(quant_files, type = "salmon", 
                   txOut = FALSE, tx2gene = tx_map, countsFromAbundance = "lengthScaledTPM")

In [ ]:
class(txi.tx)
names(txi.tx)
head(txi.tx$counts)

In [ ]:
#change column names to match metafile
col.names <- colnames(txi.tx$counts)
newnames <- gsub('./discovery_validation/quant_files/', '', col.names)
newnames <- gsub('_quant.sf', '', newnames)
head(newnames)
length(newnames)

In [ ]:
colnames(txi.tx$counts) <- newnames
head(txi.tx$counts)

In [ ]:
counts <- txi.tx$counts
write.csv(counts, file = "discovery_validation/discovery_validation_samplecounts_20260105.csv", )

# Normalization

In [ ]:
library(limma)
library(edgeR)
library(dplyr)
library(gdata)

## Discovery

In [ ]:
name <- "PE_onset_salmon_gene"
name2 <- "20260112"
batch_name <- "Disc"
batch_name2 <- "discovery"

In [ ]:
#read in metafile
meta <- read.csv(paste0("discovery_validation/", batch_name, "_cohort_metadata_20251219.csv"), 
                 stringsAsFactors=F, row.names = 1)
row.names(meta) <- meta$SampleID

head(meta)

In [ ]:
#subset counts to match metafile
meta_new <- subset(meta, variant_cohort == batch_name2)

counts <- read.csv("discovery_validation/discovery_validation_samplecounts_20260105.csv", row.names = 1)
counts_sub <- counts[,meta_new$SampleID]
dim(counts_sub)
head(counts_sub)

counts <- counts_sub

In [ ]:
#Sort metadata to match order of sample names in GeneCounts

geneCount_names_order <- colnames(counts)
meta_new$SampleID <- reorder.factor(meta_new$SampleID, new.order=geneCount_names_order)
meta_new <- meta_new %>% arrange(SampleID)
detach("package:dplyr", unload=TRUE)
head(meta_new)
head(counts)

In [ ]:
library(mygene)
#geneid <- sapply(strsplit(rownames(counts), ".", fixed=T), function(x) x[1])
geneid <- rownames(counts)

genes<- queryMany(geneid, scopes=c("ensembl.gene"), 
                  fields=c("symbol", "entrezgene"), 
                  species="human", returnall=FALSE)

In [ ]:
head(genes)

In [ ]:
row.names(genes)<-genes$query
counts2<-counts
row.names(counts2)<-geneid
counts1 <- counts2[,1:2]
genes1 <- merge(genes, counts1, by="row.names", all.x=F, all.y=T)
genes <- genes1[,c("query","entrezgene","symbol")]
colnames(genes)<- c("ENSEMBL","ENTREZID","SYMBOL")
head(genes)

In [ ]:
head(counts)
dim(counts)

In [ ]:
#Sort metadata to match order of sample names in GeneCounts\
library(dplyr)
library(gdata)

geneCount_names_order <- row.names(counts)
genes2<-as.data.frame(genes)
genes2$ENSEMBL <- reorder.factor(genes2$ENSEMBL, new.order=geneCount_names_order)
genes2 <- genes2 %>% arrange(ENSEMBL)
detach("package:dplyr", unload=TRUE)
head(genes2)
head(counts)

In [ ]:
dim(meta_new)
dim(counts)

In [ ]:
#Create DGE list
x <- DGEList(counts = counts, lib.size = colSums(counts),
norm.factors = rep(1,ncol(counts)), samples = meta_new,
group = meta_new$disease_uniform, genes = genes2, remove.zeros = FALSE)
class(x)
dim(counts)

In [ ]:
write.csv(x$samples, paste0("discovery_validation/", batch_name, "_metadata_20260105.csv"))

In [ ]:
#Estimate biological coefficient of variation (BCV)
hl <- estimateGLMCommonDisp(x, verbose=TRUE)

In [ ]:
#RNASeqPower - set depth to desired sequencing depth in millions and cv based on BCV in cell above
#BiocManager::install("RNASeqPower")
library(RNASeqPower)
rnapower(depth=20, cv=0.384 , effect=c(1.25, 1.5, 1.75, 2),
alpha= .05, power=c(.8, .9))

In [ ]:
#Plot library size in millions of reads for each sample
options(scipen=-1)
barplot(x$samples$lib.size, horiz=F, names.arg=colnames(x$counts), las=2,
       main="Library size per sample")

In [ ]:
cpm <- cpm(x)
lcpm <- cpm(x, log=TRUE)
keep.exprs<-rowSums(cpm>1) >= nrow(meta_new)/2 #at least one count in half of the samples
x <- x[keep.exprs,, keep.lib.sizes=FALSE]
dim(x)

x <- calcNormFactors(x, method = "TMM")

In [ ]:
library(RColorBrewer)
nsamples <- ncol(x)
col <- brewer.pal(nsamples, "Paired")
#par(mfrow=c(1,2))
plot(density(lcpm[,1]), col=col[1], lwd=2, ylim=c(0,.8), las=2, 
     main="", xlab="")
title(main="A. Raw data", xlab="Log-cpm")
abline(v=0, lty=3)
for (i in 2:nsamples){
 den <- density(lcpm[,i])
 lines(den$x, den$y, col=col[i], lwd=2)
}
#legend("topright", colnames(x), text.col=col, bty="n")
lcpm <- cpm(x, log=TRUE)
plot(density(lcpm[,1]), col=col[1], lwd=2, ylim=c(0,.8), las=2, 
     main="", xlab="")
title(main="B. Filtered data", xlab="Log-cpm")
abline(v=0, lty=3)
for (i in 2:nsamples){
   den <- density(lcpm[,i])
   lines(den$x, den$y, col=col[i], lwd=2)
}
#legend("topright", colnames(x), text.col=col, bty="n")

lcpm <- cpm(x, log=TRUE)
plot(density(lcpm[,1]), col=col[1], lwd=2, ylim=c(0,.8), las=2, 
     main="", xlab="")
title(main="B. Filtered data", xlab="Log-cpm")
abline(v=0, lty=3)
for (i in 2:nsamples){
   den <- density(lcpm[,i])
   lines(den$x, den$y, col=col[i], lwd=2)
}
legend("topright", colnames(x), text.col=col, bty="n")

In [ ]:
x2 <- x
x2$samples$norm.factors <- 1
x2$counts[,1] <- ceiling(x2$counts[,1]*0.05)
x2$counts[,2] <- x2$counts[,2]*5
#par(mfrow=c(1,2))
lcpm <- cpm(x2, log=TRUE)
boxplot(lcpm, las=2, col=col, main="")
title(main="A.  Unnormalized data",ylab="Log-cpm")
x2 <- calcNormFactors(x2)  

lcpm <- cpm(x2, log=TRUE)
boxplot(lcpm, las=2, col=col, main="")
title(main="B. Normalized data",ylab="Log-cpm")

In [ ]:
#pdf("mds_plot_small_labels.pdf")
#plotMDS(lcpm, labels=x$samples$SampleID, col=col.1, cex=.25)
#    title(main=i)
#dev.off()

In [ ]:
# #Create MDS plots
# variablelist <- c('PE_binary','transforming_growth_factor_beta_production_pathway')
# lcpm <- cpm(x, log=TRUE)
# for (i in variablelist){
#     new_factor <- as.factor(x$samples[,i])
#     col.1 <- new_factor
#     char.1 <- new_factor
#     levels(col.1)<-c('#E41A1C','#377EB8','#4DAF4A','#984EA3','#FF7F00','#FFFF33','#A65628','#F781BF','#999999','#66C2A5','#FC8D62','#8DA0CB','#E78AC3','#A6D854','#FFD92F','#E5C494','#B3B3B3')
#     #levels(col.1) <-  brewer.pal(nlevels(col.1), "Set1")
#     col.1 <- as.character(col.1)
#     levels(char.1) <- c(0:nlevels(char.1))
#     plotMDS(lcpm, labels=new_factor, col=col.1, cex=.75)
#     title(main=i)
#     plotMDS(lcpm, labels=x$samples$Sample, col=col.1, cex=.5)
#     title(main=i)
#     plotMDS(lcpm, pch=19, col=col.1, cex=.75)
#     title(main=i)
# }

# pdf("Variants_RNAseq_MDS_plots_20240328_genes_circulating.pdf")
# for (i in variablelist){
#     new_factor <- as.factor(x$samples[,i])
#     col.1 <- new_factor
#     char.1 <- new_factor
#     levels(col.1)<-c('#E41A1C','#377EB8','#4DAF4A','#984EA3','#FF7F00','#FFFF33','#A65628','#F781BF','#999999','#66C2A5','#FC8D62','#8DA0CB','#E78AC3','#A6D854','#FFD92F','#E5C494','#B3B3B3')
#     #levels(col.1) <-  brewer.pal(nlevels(col.1), "Set1")
#     col.1 <- as.character(col.1)
#     levels(char.1) <- c(0:nlevels(char.1))
#     plotMDS(lcpm, labels=new_factor, col=col.1, cex=.75)
#     title(main=i)
#     plotMDS(lcpm, labels=x$samples$Sample, col=col.1, cex=.75)
#     title(main=i)
#     plotMDS(lcpm, pch=19, col=col.1, cex=.75)
#     title(main=i)
# }
# dev.off()

In [ ]:
# #Interactive MDS plot
# library(Glimma)
# glMDSPlot(lcpm, 
#           groups=x$samples, html="MDSplot_salmon_genes_circulating")

In [ ]:
PE_onset_uniform <- as.factor(x$samples$PE_onset_uniform)
fetal_sex <- as.factor(x$samples$Fetal_sex_binary)
batch <- as.factor(x$samples$batch.x)


design <- model.matrix(~0+PE_onset_uniform+fetal_sex+batch, data = x$samples)
colnames(design) <- gsub("PE_onset_uniform","", colnames(design))
head(design)

In [ ]:
cm <- makeContrasts(
    Normal_LOvsEO=Normal_LO-Normal_EO,
    PE_LOvsEO=PE_LO-PE_EO,
    LOvsEO_PEvsNormal = (PE_LO-PE_EO)-(Normal_LO-Normal_EO),
    LO_PEvsNormal=PE_LO-Normal_LO,
    EO_PEvsNormal=PE_EO-Normal_EO,
    PEvsNormal_LOvsEO = (PE_LO-Normal_LO)-(PE_EO-Normal_EO),
    allPEvsNormal = (PE_LO+PE_EO)/2-(Normal_LO+Normal_EO)/2,
levels=design)

In [ ]:
v <- voom(x, design, plot=TRUE)

In [ ]:
#input the inter-subject correlation into the linear model fit
#replace x$count with v after voom
vfit <- lmFit(v,design)

vfit <- contrasts.fit(vfit, contrasts=cm)
efit <- eBayes(vfit)
plotSA(efit, main="Final model: Mean-variance trend")

In [ ]:
#DE genes
dt <- decideTests(efit)
summary(dt)

In [ ]:
#DE genes strict
dt <- decideTests(efit, lfc=1)
summary(dt)

In [ ]:
write.csv(v$E, paste0("discovery_validation/", batch_name, "_TMM_voom_normalized_counts_20260105.csv"))

In [ ]:
contrast_list <- colnames(cm)

for (i in contrast_list){
    df <- topTreat(efit, coef=i, n=Inf, sort.by="p")
    df$ENSEMBL <- vapply(df$ENSEMBL, paste, collapse = ", ", character(1L))

    print(i)
    print(head(df))
    write.csv(df, sprintf("discovery_validation/%s_DE_Coeff_%s_%s_%s.csv", batch_name, i, name, name2))}

In [ ]:
# split by pathology
fetal_sex <- as.factor(x$samples$Fetal_sex_binary)
batch <- as.factor(x$samples$batch.x)
path <-as.factor(x$samples$asw_lesion_category)

design <- model.matrix(~0+path+fetal_sex+batch, data = x$samples)
colnames(design) <- gsub("path","", colnames(design))
colnames(design) <- gsub("\\+","and", colnames(design))
colnames(design) <- gsub(" ","", colnames(design))
colnames(design) <- gsub("-","", colnames(design))


head(design)

In [ ]:
cm <- makeContrasts(
    MVMvsNormal = (MVMandFVM+MVMandVUE+pureMVM)/3-nonlesion,
    FVMvsNormal = (MVMandFVM+pureFVM)/2-nonlesion,
    MVMvsFVM=(MVMandFVM+MVMandVUE+pureMVM)/3-(MVMandFVM+pureFVM)/2,
    VillitisvsNormal =(MVMandVUE+pureVUE)/2-nonlesion,
    MVMorFVMvsNormal=(MVMandFVM+MVMandVUE+pureMVM+pureFVM)/4-nonlesion,
    LesionvsNormal=(MVMandFVM+MVMandVUE+pureMVM+pureFVM+pureVUE)/5-nonlesion,
    
levels=design)

In [ ]:
v <- voom(x, design, plot=TRUE)

In [ ]:
#input the inter-subject correlation into the linear model fit
#replace x$count with v after voom
vfit <- lmFit(v,design)

vfit <- contrasts.fit(vfit, contrasts=cm)
efit <- eBayes(vfit)
plotSA(efit, main="Final model: Mean-variance trend")

In [ ]:
#DE genes
dt <- decideTests(efit)
summary(dt)

In [ ]:
#DE genes strict
dt <- decideTests(efit, lfc=1)
summary(dt)

In [ ]:
write.csv(v$E, paste0("discovery_validation/", batch_name, "_path_TMM_voom_normalized_counts_20260105.csv"))

In [ ]:
contrast_list <- colnames(cm)

for (i in contrast_list){
    df <- topTreat(efit, coef=i, n=Inf, sort.by="p")
    df$ENSEMBL <- vapply(df$ENSEMBL, paste, collapse = ", ", character(1L))

    print(i)
    print(head(df))
    write.csv(df, sprintf("discovery_validation/%s_pathology_DE_Coeff_%s_%s_%s.csv", batch_name, i, name, name2))}

## Validation

In [ ]:
library(dplyr)

In [ ]:
name <- "PE_onset_salmon_gene"
name2 <- "20260112"
batch_name <- "Val"
batch_name2 <- "validation"

In [ ]:
#read in metafile
meta <- read.csv(paste0("discovery_validation/", batch_name, "_cohort_metadata_20251219.csv"), 
                 stringsAsFactors=F, row.names = 1)
row.names(meta) <- meta$SampleID

head(meta)

In [ ]:
#subset counts to match metafile
meta_new <- subset(meta, variant_cohort == batch_name2)

counts <- read.csv("discovery_validation/discovery_validation_samplecounts_20260105.csv", row.names = 1)
counts_sub <- counts[,meta_new$SampleID]
dim(counts_sub)
head(counts_sub)

counts <- counts_sub

In [ ]:
#Sort metadata to match order of sample names in GeneCounts

geneCount_names_order <- colnames(counts)
meta_new$SampleID <- reorder.factor(meta_new$SampleID, new.order=geneCount_names_order)
meta_new <- meta_new %>% arrange(SampleID)
detach("package:dplyr", unload=TRUE)
head(meta_new)
head(counts)

In [ ]:
library(mygene)
#geneid <- sapply(strsplit(rownames(counts), ".", fixed=T), function(x) x[1])
geneid <- rownames(counts)

genes<- queryMany(geneid, scopes=c("ensembl.gene"), 
                  fields=c("symbol", "entrezgene"), 
                  species="human", returnall=FALSE)

In [ ]:
head(genes)

In [ ]:
row.names(genes)<-genes$query
counts2<-counts
row.names(counts2)<-geneid
counts1 <- counts2[,1:2]
genes1 <- merge(genes, counts1, by="row.names", all.x=F, all.y=T)
genes <- genes1[,c("query","entrezgene","symbol")]
colnames(genes)<- c("ENSEMBL","ENTREZID","SYMBOL")
head(genes)

In [ ]:
head(counts)
dim(counts)

In [ ]:
#Sort metadata to match order of sample names in GeneCounts\
library(dplyr)
library(gdata)

geneCount_names_order <- row.names(counts)
genes2<-as.data.frame(genes)
genes2$ENSEMBL <- reorder.factor(genes2$ENSEMBL, new.order=geneCount_names_order)
genes2 <- genes2 %>% arrange(ENSEMBL)
detach("package:dplyr", unload=TRUE)
head(genes2)
head(counts)

In [ ]:
dim(meta_new)
dim(counts)

In [ ]:
#Create DGE list
x <- DGEList(counts = counts, lib.size = colSums(counts),
norm.factors = rep(1,ncol(counts)), samples = meta_new,
group = meta_new$disease_uniform, genes = genes2, remove.zeros = FALSE)
class(x)
dim(counts)

In [ ]:
write.csv(x$samples, paste0("discovery_validation/", batch_name, "_metadata_20260105.csv"))

In [ ]:
#Estimate biological coefficient of variation (BCV)
hl <- estimateGLMCommonDisp(x, verbose=TRUE)

In [ ]:
#RNASeqPower - set depth to desired sequencing depth in millions and cv based on BCV in cell above
#BiocManager::install("RNASeqPower")
library(RNASeqPower)
rnapower(depth=20, cv=0.384 , effect=c(1.25, 1.5, 1.75, 2),
alpha= .05, power=c(.8, .9))

In [ ]:
#Plot library size in millions of reads for each sample
options(scipen=-1)
barplot(x$samples$lib.size, horiz=F, names.arg=colnames(x$counts), las=2,
       main="Library size per sample")

In [ ]:
cpm <- cpm(x)
lcpm <- cpm(x, log=TRUE)
keep.exprs<-rowSums(cpm>1)> nrow(meta_new)/2 #at least one count in half of the samples
x <- x[keep.exprs,, keep.lib.sizes=FALSE]
dim(x)

x <- calcNormFactors(x, method = "TMM")

In [ ]:
library(RColorBrewer)
nsamples <- ncol(x)
col <- brewer.pal(nsamples, "Paired")
#par(mfrow=c(1,2))
plot(density(lcpm[,1]), col=col[1], lwd=2, ylim=c(0,.8), las=2, 
     main="", xlab="")
title(main="A. Raw data", xlab="Log-cpm")
abline(v=0, lty=3)
for (i in 2:nsamples){
 den <- density(lcpm[,i])
 lines(den$x, den$y, col=col[i], lwd=2)
}
#legend("topright", colnames(x), text.col=col, bty="n")
lcpm <- cpm(x, log=TRUE)
plot(density(lcpm[,1]), col=col[1], lwd=2, ylim=c(0,.8), las=2, 
     main="", xlab="")
title(main="B. Filtered data", xlab="Log-cpm")
abline(v=0, lty=3)
for (i in 2:nsamples){
   den <- density(lcpm[,i])
   lines(den$x, den$y, col=col[i], lwd=2)
}
#legend("topright", colnames(x), text.col=col, bty="n")

lcpm <- cpm(x, log=TRUE)
plot(density(lcpm[,1]), col=col[1], lwd=2, ylim=c(0,.8), las=2, 
     main="", xlab="")
title(main="B. Filtered data", xlab="Log-cpm")
abline(v=0, lty=3)
for (i in 2:nsamples){
   den <- density(lcpm[,i])
   lines(den$x, den$y, col=col[i], lwd=2)
}
legend("topright", colnames(x), text.col=col, bty="n")

In [ ]:
x2 <- x
x2$samples$norm.factors <- 1
x2$counts[,1] <- ceiling(x2$counts[,1]*0.05)
x2$counts[,2] <- x2$counts[,2]*5
#par(mfrow=c(1,2))
lcpm <- cpm(x2, log=TRUE)
boxplot(lcpm, las=2, col=col, main="")
title(main="A.  Unnormalized data",ylab="Log-cpm")
x2 <- calcNormFactors(x2)  

lcpm <- cpm(x2, log=TRUE)
boxplot(lcpm, las=2, col=col, main="")
title(main="B. Normalized data",ylab="Log-cpm")

In [ ]:
#pdf("mds_plot_small_labels.pdf")
#plotMDS(lcpm, labels=x$samples$SampleID, col=col.1, cex=.25)
#    title(main=i)
#dev.off()

In [ ]:
# #Create MDS plots
# variablelist <- c('PE_binary','transforming_growth_factor_beta_production_pathway')
# lcpm <- cpm(x, log=TRUE)
# for (i in variablelist){
#     new_factor <- as.factor(x$samples[,i])
#     col.1 <- new_factor
#     char.1 <- new_factor
#     levels(col.1)<-c('#E41A1C','#377EB8','#4DAF4A','#984EA3','#FF7F00','#FFFF33','#A65628','#F781BF','#999999','#66C2A5','#FC8D62','#8DA0CB','#E78AC3','#A6D854','#FFD92F','#E5C494','#B3B3B3')
#     #levels(col.1) <-  brewer.pal(nlevels(col.1), "Set1")
#     col.1 <- as.character(col.1)
#     levels(char.1) <- c(0:nlevels(char.1))
#     plotMDS(lcpm, labels=new_factor, col=col.1, cex=.75)
#     title(main=i)
#     plotMDS(lcpm, labels=x$samples$Sample, col=col.1, cex=.5)
#     title(main=i)
#     plotMDS(lcpm, pch=19, col=col.1, cex=.75)
#     title(main=i)
# }

# pdf("Variants_RNAseq_MDS_plots_20240328_genes_circulating.pdf")
# for (i in variablelist){
#     new_factor <- as.factor(x$samples[,i])
#     col.1 <- new_factor
#     char.1 <- new_factor
#     levels(col.1)<-c('#E41A1C','#377EB8','#4DAF4A','#984EA3','#FF7F00','#FFFF33','#A65628','#F781BF','#999999','#66C2A5','#FC8D62','#8DA0CB','#E78AC3','#A6D854','#FFD92F','#E5C494','#B3B3B3')
#     #levels(col.1) <-  brewer.pal(nlevels(col.1), "Set1")
#     col.1 <- as.character(col.1)
#     levels(char.1) <- c(0:nlevels(char.1))
#     plotMDS(lcpm, labels=new_factor, col=col.1, cex=.75)
#     title(main=i)
#     plotMDS(lcpm, labels=x$samples$Sample, col=col.1, cex=.75)
#     title(main=i)
#     plotMDS(lcpm, pch=19, col=col.1, cex=.75)
#     title(main=i)
# }
# dev.off()

In [ ]:
# #Interactive MDS plot
# library(Glimma)
# glMDSPlot(lcpm, 
#           groups=x$samples, html="MDSplot_salmon_genes_circulating")

In [ ]:
PE_onset_uniform <- as.factor(x$samples$PE_onset_uniform)
fetal_sex <- as.factor(x$samples$Fetal_sex_binary)


design <- model.matrix(~0+PE_onset_uniform+fetal_sex, data = x$samples)
colnames(design) <- gsub("PE_onset_uniform","", colnames(design))
head(design)

In [ ]:
cm <- makeContrasts(
    PE_LOvsEO=PE_LO-PE_EO,
    LO_PEvsNormal=PE_LO-Normal_LO,
    levels=design)

In [ ]:
v <- voom(x, design, plot=TRUE)

In [ ]:
#input the inter-subject correlation into the linear model fit
#replace x$count with v after voom
vfit <- lmFit(v,design)

vfit <- contrasts.fit(vfit, contrasts=cm)
efit <- eBayes(vfit)
plotSA(efit, main="Final model: Mean-variance trend")

In [ ]:
#DE genes
dt <- decideTests(efit)
summary(dt)

In [ ]:
#DE genes strict
dt <- decideTests(efit, lfc=1)
summary(dt)

In [ ]:
write.csv(v$E, paste0("discovery_validation/", batch_name, "_TMM_voom_normalized_counts_20260105.csv"))

In [ ]:
contrast_list <- colnames(cm)

for (i in contrast_list){
    df <- topTreat(efit, coef=i, n=Inf, sort.by="p")
    df$ENSEMBL <- vapply(df$ENSEMBL, paste, collapse = ", ", character(1L))

    print(i)
    print(head(df))
    write.csv(df, sprintf("discovery_validation/%s_DE_Coeff_%s_%s_%s.csv", batch_name, i, name, name2))}